In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%pip -q install --force-reinstall "streamlit==1.32.2" "protobuf<5"
import os; print("🔄 Restarting runtime…"); os.kill(os.getpid(), 9)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.2/731.2 kB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.3/107.3 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.2/208.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 113.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.4/242.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 443.9/443.9 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# --- Clean + install an ABI-safe stack for TF 2.17 on Colab ---
%pip -q uninstall -y numpy jax jaxlib gradio gradio_client cloudflared || true
%pip -q install numpy==1.26.4 pandas==2.2.2 tensorflow==2.17.0 opencv-python-headless==4.10.0.84 streamlit==1.38.0 mtcnn==0.1.1

# System codecs for mp4/mov
!apt-get -qq update && apt-get -qq install -y ffmpeg libsm6 libxext6

# ---- Install cloudflared as a standalone binary (more reliable on Colab) ----
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version | head -n 1

# Print versions we will use AFTER the restart
import numpy as np, tensorflow as tf, cv2, sys
print("✅ Will use -> NumPy", np.__version__, "| TF", tf.__version__, "| OpenCV", cv2.__version__, "| Python", sys.version.split()[0])

# 🔄 IMPORTANT: Force a clean runtime restart so NumPy/TF C-extensions reload correctly.
import os
print("🔄 Restarting runtime to finalize installs…")
os.kill(os.getpid(), 9)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 81.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires jax>=0.1.72, which is not installed.
dopamine-rl 4.1.2 requires jaxlib>=0.1.51, which is not installed.
flax 0.10.7 requires jax>=0.6.0, which is not installed.
orbax-checkpoint 0.11.25 requires jax>=0.6.0, which is not installed.
chex 0.1.90 requires jax>=0.4.27, which is not installed.
chex 0.1.90 requires jaxlib>=0.4.27, which is not installed.
optax 0.2.6 requires jax>=0.5.3, which is not installed.
optax 0.2.6 requires jaxlib>=0.5.3, which is not installed.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
google-colab 1.0.0 requires tornado==6.5.1, but you have tornado 6.5.2 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26

In [10]:
# 👉 Set to the exact path of your Keras .h5 weights (on Drive or /content)
# Examples:
# WEIGHTS_PATH = "/content/xception_ffpp_weights.h5"
# WEIGHTS_PATH = "/content/drive/MyDrive/xception_ffpp_weights.h5"
WEIGHTS_PATH = "/content/drive/MyDrive/xception_ffpp_weights.h5"   # <— EDIT THIS

import os, glob
# common typo auto-fix
if not os.path.exists(WEIGHTS_PATH):
    typo = "/content/xception_ffpp_weigths.h5"
    if os.path.exists(typo):
        import shutil; shutil.move(typo, "/content/xception_ffpp_weights.h5")
        WEIGHTS_PATH = "/content/xception_ffpp_weights.h5"

print("Weights path set to:", WEIGHTS_PATH)
print("Exists:", os.path.exists(WEIGHTS_PATH))

# If False, search Drive for likely candidates:
# !find /content/drive -iname "*xception*weights*.h5"


Weights path set to: /content/drive/MyDrive/xception_ffpp_weights.h5
Exists: True


In [11]:
# ===== Cell 4 (fixed) — write Streamlit app and pass weights path =====

app_py = '''
import os, time, cv2, numpy as np, streamlit as st, tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import xception as xcep
from pathlib import Path

# --------- Config ---------
WEIGHTS_PATH = os.environ.get("DF_WEIGHTS", "/content/drive/MyDrive/xception_ffpp_weights.h5")
IMG_SIZE     = (299, 299)
FRAME_RATE   = 3
ENLARGE      = 1.3
TOP_SHOW     = 6
THRESHOLD    = 0.5

# Safer GPU usage
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
for g in tf.config.list_physical_devices("GPU"):
    try: tf.config.experimental.set_memory_growth(g, True)
    except: pass

# --------- Model ---------
@st.cache_resource(show_spinner=False)
def load_model(weights_path):
    if not Path(weights_path).exists():
        raise FileNotFoundError(f"Weights not found at: {weights_path}")
    try:
        m = tf.keras.models.load_model(weights_path, compile=False)
        return m
    except Exception as e:
        # Fallback: build Xception head and load weights
        base = xcep.Xception(include_top=False, weights=None, input_shape=(299,299,3))
        x = layers.GlobalAveragePooling2D()(base.output)
        out = layers.Dense(1, activation="sigmoid")(x)
        m = models.Model(base.input, out)
        m.load_weights(weights_path)
        return m

model = load_model(WEIGHTS_PATH)

# --------- Haar face detector ---------
haar_xml = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
FACE_DET = cv2.CascadeClassifier(haar_xml)
if FACE_DET.empty():
    st.error("Failed to load Haar cascade. Restart runtime and run installs again.")
    st.stop()

def detect_faces_rgb(rgb):
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    faces = FACE_DET.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60,60))
    return [[int(x), int(y), int(w), int(h)] for (x,y,w,h) in faces]

def extract_face_crops(video_path, fps=FRAME_RATE, enlarge=ENLARGE, max_faces=1200):
    faces = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        fixed = "/conte nt/_fixed_input.mp4"
        os.system(f'ffmpeg -y -i "{video_path}" -vcodec libx264 -acodec aac "{fixed}" >/dev/null 2>&1')
        cap = cv2.VideoCapture(fixed)
        if not cap.isOpened():
            raise RuntimeError("Video cannot be opened. Please upload a short H.264 .mp4/.mov.")
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(native_fps / fps)), 1)
    idx = taken = 0
    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            for (x, y, w, h) in detect_faces_rgb(rgb):
                cx, cy = x + w//2, y + h//2
                nw, nh = int(w*enlarge), int(h*enlarge)
                x1, y1 = max(0, cx-nw//2), max(0, cy-nh//2)
                x2, y2 = min(rgb.shape[1], cx+nw//2), min(rgb.shape[0], cy+nh//2)
                crop = rgb[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop = cv2.resize(crop, IMG_SIZE)
                faces.append(crop)
                taken += 1
                if taken >= max_faces: break
        idx += 1
        if taken >= max_faces: break
    cap.release()
    return faces

# --------- UI ---------
st.set_page_config(page_title="Deepfake Checker (XceptionNet)", page_icon="🎬", layout="wide")
st.title("🎬 Deepfake Checker (XceptionNet)")
st.caption("Upload a short video; the app samples frames, detects faces, and predicts Real/Fake.")

with st.sidebar:
    st.subheader("Speed & Threshold")
    fps = st.slider("Sampling FPS", 1, 6, 3, 1)
    max_faces = st.number_input("Max faces to score", 50, 2000, 400, 50)
    THRESHOLD = st.slider("Decision threshold (Fake ≥)", 0.0, 1.0, 0.5, 0.01)
    agg = st.selectbox("Aggregate frame scores by", ["mean","median"])

uploaded = st.file_uploader("Upload a video (.mp4 / .mov)", type=["mp4","mov"])
if uploaded is None:
    st.info("Upload a short video to begin.")
    st.stop()

tmp_path = "/content/_upload.mp4"
with open(tmp_path, "wb") as f:
    f.write(uploaded.read())
st.video(tmp_path)

# —— fast detection: run Haar on a smaller copy, crop from full-res —— #
def extract_face_crops_fast(video_path, fps=3, enlarge=1.3, max_faces=400, min_size=60):
    faces = []
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        fixed = "/content/_fixed_input.mp4"
        os.system(f'ffmpeg -y -i "{video_path}" -vcodec libx264 -acodec aac "{fixed}" >/dev/null 2>&1')
        cap = cv2.VideoCapture(fixed)
        if not cap.isOpened():
            raise RuntimeError("Video cannot be opened. Please upload a short H.264 .mp4/.mov.")

    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(native_fps / fps)), 1)

    idx = taken = 0
    pbar = st.progress(0, text="Extracting faces...")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1

    while True:
        ok, frame = cap.read()
        if not ok: break
        if idx % step == 0:
            H, W = frame.shape[:2]
            # build a small preview for detection
            target_w = 640
            scale = target_w / float(W)
            small = cv2.resize(frame, (target_w, int(H*scale)))
            small_rgb = cv2.cvtColor(small, cv2.COLOR_BGR2RGB)
            # detect on small
            gray = cv2.cvtColor(small_rgb, cv2.COLOR_RGB2GRAY)
            dets = FACE_DET.detectMultiScale(gray, 1.1, 5, minSize=(min_size, min_size))
            # map boxes back to full-res and crop
            for (sx, sy, sw, sh) in dets:
                x, y, w, h = int(sx/scale), int(sy/scale), int(sw/scale), int(sh/scale)
                cx, cy = x + w//2, y + h//2
                nw, nh = int(w*enlarge), int(h*enlarge)
                x1, y1 = max(0, cx-nw//2), max(0, cy-nh//2)
                x2, y2 = min(W, cx+nw//2), min(H, cy+nh//2)
                crop = frame[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
                crop = cv2.resize(crop, IMG_SIZE)
                faces.append(crop)
                taken += 1
                if taken >= max_faces:
                    break
        idx += 1
        if idx % 30 == 0:
            pbar.progress(min(idx/total, 1.0))
        if taken >= max_faces:
            break

    cap.release()
    pbar.progress(1.0)
    return faces

with st.spinner("Processing…"):
    t0 = time.time()
    try:
        crops = extract_face_crops_fast(tmp_path, fps=fps, enlarge=ENLARGE, max_faces=max_faces)
    except Exception as e:
        st.error(f"Could not read/process the video: {e}")
        st.stop()

    if len(crops) == 0:
        st.warning("No face detected – please upload a clearer video.")
        st.stop()

    x = xcep.preprocess_input(np.asarray(crops, dtype=np.float32))
    probs = model.predict(x, verbose=0).ravel()
    prob_fake = float(np.mean(probs) if agg=="mean" else np.median(probs))
    label = "Fake" if prob_fake >= THRESHOLD else "Real"
    elapsed = time.time() - t0

color = "#ff4b4b" if label=="Fake" else "#4CAF50"
st.markdown(f"""<div style='padding:10px;border-radius:8px;background:{color};color:white;
                display:inline-block;font-weight:700'>
                Prediction: {label} — Confidence: {prob_fake:.2f}</div>""",
            unsafe_allow_html=True)
st.write(f"**Frames analyzed:** {len(crops)}")
st.write(f"**Processing time:** {elapsed:.1f}s")

order = np.argsort(probs)[::-1]
top_idx = order[:min(TOP_SHOW, len(crops))]
st.subheader("Top frames")
cols = st.columns(min(3, max(1, len(top_idx))))
for i, idx in enumerate(top_idx):
    with cols[i % len(cols)]:
        st.image(crops[idx], caption=f"prob_fake={probs[idx]:.2f}", use_column_width=True)

'''

# Write app file
with open("app.py", "w") as f:
    f.write(app_py)

# Pass weights path to the app via env var
import os
os.environ["DF_WEIGHTS"] = WEIGHTS_PATH
print("app.py written. DF_WEIGHTS ->", os.environ["DF_WEIGHTS"])


app.py written. DF_WEIGHTS -> /content/drive/MyDrive/xception_ffpp_weights.h5


In [12]:
# Kill any previous Streamlit
!ps -ax | grep "streamlit run app.py" | grep -v grep | awk '{print $1}' | xargs -r kill -9

import subprocess, time, re, requests

# Start Streamlit (extra flags help behind tunnels)
args = [
    "streamlit","run","app.py",
    "--server.address","0.0.0.0",
    "--server.port","8081",
    "--server.headless","true",
    "--server.enableXsrfProtection","false",
    "--browser.gatherUsageStats","false",
    "--server.enableWebsocketCompression","false",
    "--server.maxUploadSize","500",     # allow up to 500 MB
    "--server.fileWatcherType","none",  # fewer FS events via tunnel
]
srv = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait for Streamlit to be healthy
for _ in range(60):
    try:
        if "ok" in requests.get("http://127.0.0.1:8081/_stcore/health", timeout=1).text:
            break
    except Exception:
        pass
    time.sleep(1)

# Add a short buffer to ensure session is ready before proxy connects
time.sleep(2.0)

# Start cloudflared tunnel
print("Opening cloudflared tunnel…")
tunnel = subprocess.Popen(
    ["cloudflared","tunnel","--url","http://localhost:8081","--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

public_url = None
for _ in range(120):
    line = tunnel.stdout.readline().strip()
    if line:
        print(line)
        m = re.search(r"(https://[-a-z0-9.]+trycloudflare.com)", line)
        if m:
            public_url = m.group(1); break
    time.sleep(0.25)

print("\n✅ Open your Streamlit app:", public_url or "(scroll logs for trycloudflare link)")


Opening cloudflared tunnel…
2025-10-24T20:43:38Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2025-10-24T20:43:38Z INF Requesting new quick Tunnel on trycloudflare.com...
2025-10-24T20:43:42Z INF +--------------------------------------------------------------------------------------------+
2025-10-24T20:43:42Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2025-10-24T20:43:42Z INF |  https://halloween-bacteri